# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [3]:
# from langchain_community.document_loaders.csv_loader import CSVLoader
# from datetime import datetime, timedelta

# loader = CSVLoader(
#     file_path=f"./data/Projects_with_Domains.csv",
#     metadata_columns=[
#       "Project Title",
#       "Project Domain",
#       "Secondary Domain",
#       "Description",
#       "Judge Comments",
#       "Score",
#       "Project Name",
#       "Judge Score"
#     ]
# )

# synthetic_usecase_data = loader.load()

# for doc in synthetic_usecase_data:
#     doc.page_content = doc.metadata["Description"]

In [4]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/arxiv_cleaned_sample.csv",
    metadata_columns=[
      "id",
      "title",
      "abstract",
      "categories",
      "authors",
    ],
    source_column="combined_text"
)

synthetic_arxiv_data = loader.load()


Let's look at an example document to see if everything worked as expected!

In [5]:
synthetic_arxiv_data[0]

Document(metadata={'source': "Title: Calculation of prompt diphoton production cross sections at Tevatron and LHC energies. Authors: C. Bal\\'azs, E. L. Berger, P. M. Nadolsky, C.-P. Yuan. Categories: hep-ph. Abstract: A fully differential calculation in perturbative quantum chromodynamics is presented for the production of massive photon pairs at hadron colliders. All next-to-leading order perturbative contributions from quark-antiquark, gluon-(anti)quark, and gluon-gluon subprocesses are included, as well as all-orders resummation of initial-state gluon radiation valid at next-to-next-to-leading logarithmic accuracy. The region of phase space is specified in which the calculation is most reliable. Good agreement is demonstrated with data from the Fermilab Tevatron, and predictions are made for more detailed tests with CDF and DO data. Predictions are shown for distributions of diphoton pairs produced at the energy of the Large Hadron Collider (LHC). Distributions of the diphoton pair

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [6]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    synthetic_arxiv_data,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Arxiv_Data"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [7]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [9]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [10]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [11]:
naive_retrieval_chain.invoke({"question" : "Which paper discusses diphoton production or cross sections in particle physics?"})["response"].content

'Among the provided documents, several papers discuss diphoton production or related cross sections in particle physics:\n\n1. **"Calculation of prompt diphoton production cross sections at Tevatron and LHC energies"**  \n   - Focuses on a detailed QCD calculation of diphoton (massive photon pairs) production at hadron colliders, including NLO contributions and resummations.  \n   - Abstract: "A fully differential calculation in perturbative quantum chromodynamics is presented for the production of massive photon pairs at hadron colliders."\n\n2. **"Direct photons and dileptons via color dipoles"**  \n   - Discusses inclusive direct photon production, which is related to photon production cross sections, including at the Tevatron.\n\n3. **"Electromagnetic Higgs production"**  \n   - Mentions Higgs production via photon fusion, related to photon cross sections, but not specifically diphoton cross sections.\n\nThe most comprehensive paper explicitly dedicated to diphoton production cross

In [12]:
naive_retrieval_chain.invoke({"question" : "Find the paper that studies Stirling numbers or combinatorial determinants."})["response"].content

'Based on the provided information, the following papers study Stirling numbers or combinatorial determinants:\n\n1. **"A determinant of Stirling cycle numbers counts unlabeled acyclic single-source automata"** by David Callan (arXiv:0704.0004).  \n   - This paper discusses a determinant involving Stirling cycle numbers and relates it to counting unlabeled acyclic automata.\n\n2. **"Almost Product Evaluation of Hankel Determinants"** by Omer Egecioglu, Timothy Redmond, Charles Ryavec (arXiv:0704.3398).  \n   - Focuses on Hankel determinants, which are connected to combinatorial sequences and determinants evaluations, including those involving binomial coefficients that relate to Stirling-type numbers.\n\n3. **"Determinant Formulas Relating to Tableaux of Bounded Height"** by Guoce Xin (arXiv:0704.3381).  \n   - Deals with determinants associated with tableaux, which often involve combinatorial determinants and may relate to Stirling numbers.\n\n4. **"Determinant formulas relating to ta

In [13]:
naive_retrieval_chain.invoke({"question" : "Which paper investigates how the Moon’s orbit evolves over time?"})["response"].content

'The paper that investigates how the Moon’s orbit evolves over time is titled "The evolution of the Earth-Moon system based on the dark matter field fluid model" by Hongjun Pan.'

In [14]:
naive_retrieval_chain.invoke({"question" : "Which study connects harmonic analysis and Λα (Lambda-alpha) function spaces?"})["response"].content

'The study that connects harmonic analysis and Λα (Lambda-alpha) function spaces is titled "From dyadic Λα to Λα" by Wael Abu-Shammala and Alberto Torchinsky.'

In [15]:
naive_retrieval_chain.invoke({"question" : "What is the paper that introduces a sparsity-certifying algorithm for graph decomposition?"})["response"].content

'The paper that introduces a sparsity-certifying algorithm for graph decomposition is titled "Sparsity-certifying Graph Decompositions" by Ileana Streinu and Louis Theran.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [16]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(synthetic_arxiv_data)

We'll construct the same chain - only changing the retriever.

In [17]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [18]:
bm25_retrieval_chain.invoke({"question" : "Which paper discusses diphoton production or cross sections in particle physics?"})["response"].content

'The first document, titled "Calculation of prompt diphoton production cross sections at Tevatron and LHC energies," discusses diphoton production and cross sections in particle physics.'

In [19]:
bm25_retrieval_chain.invoke({"question" : "Find the paper that studies Stirling numbers or combinatorial determinants."})["response"].content

'There are several papers in the provided context that study Stirling numbers or combinatorial determinants. Here are some notable ones:\n\n1. **"A determinant of Stirling cycle numbers counts unlabeled acyclic single-source automata"** by David Callan.  \n   This paper investigates determinants involving Stirling cycle numbers and provides combinatorial interpretations related to automata theory.\n\n2. **"Combinatorics and Boson normal ordering: A gentle introduction"** by P. Blasiak, A. Horzela, K. A. Penson, A. I. Solomon, G. H. E. Duchamp.  \n   This work discusses a combinatorial framework for operator ordering problems, where the solutions involve Bell and Stirling numbers, which are key concepts in combinatorics and number theory.\n\nBoth papers study aspects of Stirling numbers within a combinatorial or algebraic context.'

In [20]:
bm25_retrieval_chain.invoke({"question" : "Which paper investigates how the Moon’s orbit evolves over time?"})["response"].content

'The paper that investigates how the Moon’s orbit evolves over time is titled "Modelling long-term trends in lunar exposure to the Earth\'s plasmasheet" by Mike Hapgood. It discusses the precession of the Moon\'s orbit around the Earth, which completes a revolution every 18.6 years and influences various astronomical phenomena, including the evolution of the Moon\'s apparent orbit over time.'

In [21]:
bm25_retrieval_chain.invoke({"question" : "Which study connects harmonic analysis and Λα (Lambda-alpha) function spaces?"})["response"].content

'I do not know of a study that specifically connects harmonic analysis and Λα (Lambda-alpha) function spaces based on the provided context.'

In [22]:
bm25_retrieval_chain.invoke({"question" : "What is the paper that introduces a sparsity-certifying algorithm for graph decomposition?"})["response"].content

'Based on the provided context, the paper that introduces a sparsity-certifying algorithm for graph decomposition is not explicitly mentioned. The documents primarily discuss algorithms related to Hamiltonian graphs, traveling salesman problems, metric dimension and diameter in graphs, anisotropic diffusion in hydrogen-bonded networks, and belief propagation for weighted matching. None of these specifically focus on a sparsity-certifying algorithm for graph decomposition.\n\nTherefore, I do not know the specific paper that introduces a sparsity-certifying algorithm for graph decomposition.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer

**Example Query:** "Which paper investigates how the Moon's orbit evolves over time?"

**Analysis of Responses:**

**Naive Retriever (Embeddings) Response:**
- Found: "The evolution of the Earth-Moon system based on the dark matter field fluid model" by Hongjun Pan
- This paper focuses on dark matter field fluid models affecting the Earth-Moon system

**BM25 Response:**
- Found: "Modelling long-term trends in lunar exposure to the Earth's plasmasheet" by Mike Hapgood
- This paper specifically discusses "the precession of the Moon's orbit around the Earth, which completes a revolution every 18.6 years and influences various astronomical phenomena, including the evolution of the Moon's apparent orbit over time"

**Why BM25 is Better:**

1. **More Direct Match to Query Intent**: The BM25 result directly addresses "how the Moon's orbit evolves over time" by discussing orbital precession and long-term orbital trends. The naive retriever found a paper about dark matter effects, which is more tangentially related.

2. **Keyword Precision**: BM25 likely matched key terms like "orbit," "Moon," and "evolution/evolves" more precisely. The query asks specifically about orbital evolution, and BM25 found a paper that explicitly discusses orbital mechanics.

3. **Temporal Focus**: The query asks about evolution "over time," and the BM25 result specifically mentions "long-term trends" and "18.6 years" cycles, showing it captured the temporal aspect better.

4. **Specificity vs. Generality**: While the embedding-based result discusses the Earth-Moon system broadly through a dark matter lens, the BM25 result is specifically focused on orbital dynamics and evolution, which is exactly what the query requested.

This demonstrates BM25's strength in matching specific technical concepts and maintaining focus on the precise aspects mentioned in the query, rather than retrieving semantically related but less directly relevant content.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [23]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [24]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [25]:
contextual_compression_retrieval_chain.invoke({"question" : "Which paper discusses diphoton production or cross sections in particle physics?"})["response"].content

'The paper that discusses diphoton production or cross sections in particle physics is titled "Calculation of prompt diphoton production cross sections at Tevatron and LHC energies," authored by C. Balázs, E. L. Berger, P. M. Nadolsky, and C.-P. Yuan. It presents a detailed calculation of diphoton production cross sections at hadron colliders, including all relevant perturbative contributions and comparisons with experimental data.'

In [26]:
contextual_compression_retrieval_chain.invoke({"question" :  "Find the paper that studies Stirling numbers or combinatorial determinants."})["response"].content

'The paper that studies Stirling numbers is titled "A determinant of Stirling cycle numbers counts unlabeled acyclic single-source automata" by David Callan.'

In [27]:
contextual_compression_retrieval_chain.invoke({"question" : "Which paper investigates how the Moon’s orbit evolves over time?"})["response"].content

'The paper titled "The evolution of the Earth-Moon system based on the dark matter field fluid model" by Hongjun Pan investigates how the Moon’s orbit evolves over time.'

In [28]:
contextual_compression_retrieval_chain.invoke({"question" : "Which study connects harmonic analysis and Λα (Lambda-alpha) function spaces?"})["response"].content

'The study that connects harmonic analysis and Λα (Lambda-alpha) function spaces is titled "From dyadic Λα to Λα" by Wael Abu-Shammala and Alberto Torchinsky.'

In [29]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the paper that introduces a sparsity-certifying algorithm for graph decomposition?"})["response"].content

'The paper that introduces a sparsity-certifying algorithm for graph decomposition is titled "Sparsity-certifying Graph Decompositions" by Ileana Streinu and Louis Theran.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [30]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [31]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [32]:
multi_query_retrieval_chain.invoke({"question" : "Which paper discusses diphoton production or cross sections in particle physics?"})["response"].content

'The paper that discusses diphoton production or cross sections in particle physics is titled "Calculation of prompt diphoton production cross sections at Tevatron and LHC energies." Its abstract indicates that it presents a fully differential calculation within perturbative QCD for massive photon pair production at hadron colliders, including all NLO contributions and all-orders resummation, with predictions and comparisons to Tevatron and LHC data.'

In [33]:
multi_query_retrieval_chain.invoke({"question" :  "Find the paper that studies Stirling numbers or combinatorial determinants."})["response"].content

'The paper that studies Stirling numbers or combinatorial determinants is titled "A determinant of Stirling cycle numbers counts unlabeled acyclic single-source automata" by David Callan.'

In [34]:
multi_query_retrieval_chain.invoke({"question" : "Which paper investigates how the Moon’s orbit evolves over time?"})["response"].content

'The paper that investigates how the Moon’s orbit evolves over time is titled "The evolution of the Earth-Moon system based on the dark matter field fluid model" by Hongjun Pan. It discusses the general pattern of the Moon-Earth system\'s evolution, including how the closest distance of the Moon to Earth was much greater in the past and how the system\'s behavior aligns with geological and fossil evidence.'

In [35]:
multi_query_retrieval_chain.invoke({"question" : "Which study connects harmonic analysis and Λα (Lambda-alpha) function spaces?"})["response"].content

'The study that connects harmonic analysis and Λα (Lambda-alpha) function spaces is titled "From dyadic Λα to Λα" by Wael Abu-Shammala and Alberto Torchinsky. This paper demonstrates how to compute the Λα norm using the dyadic grid and relates the Λα spaces to the Hardy spaces H^p(R^N) in terms of dyadic and special atoms, establishing a link between harmonic analysis techniques and the structure of Λα function spaces.'

In [36]:
multi_query_retrieval_chain.invoke({"question" : "What is the paper that introduces a sparsity-certifying algorithm for graph decomposition?"})["response"].content

'The paper that introduces a sparsity-certifying algorithm for graph decomposition is titled "Sparsity-certifying Graph Decompositions" by Ileana Streinu and Louis Theran.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer

Generating multiple reformulations of a user query improves recall through several key mechanisms:

**1. Vocabulary Mismatch Resolution**
- Users often express the same concept using different terminology than what appears in documents
- Multiple reformulations can capture alternative phrasings, synonyms, and domain-specific terminology
- Example: A user asking "machine learning algorithms" might miss documents that use "artificial intelligence methods" or "statistical learning techniques"

**2. Perspective Diversification**
- Different reformulations can approach the same topic from various angles
- This captures documents that discuss the concept from different viewpoints or contexts
- Example: Reformulating "climate change effects" into "global warming impacts," "environmental consequences," and "ecological disruption" would retrieve different but relevant document sets

**3. Specificity and Generality Balance**
- Some reformulations can be more specific while others more general
- Specific queries find highly targeted documents, while general queries catch broader relevant content
- This creates a wider net that captures both focused and contextual information

**4. Semantic Space Coverage**
- Each reformulation creates a different vector representation in embedding space
- Multiple queries explore different regions of the semantic space
- This increases the likelihood of finding relevant documents that might be semantically distant from the original query but close to a reformulation

**5. Query Ambiguity Handling**
- Original queries may be ambiguous or incomplete
- Reformulations can disambiguate by exploring different interpretations
- This ensures relevant documents aren't missed due to unclear user intent

**6. Document Diversity**
- Different reformulations may retrieve overlapping but distinct document sets
- The union of these sets provides higher recall than any single query
- This is particularly valuable when documents use varied terminology or discuss topics from different disciplinary perspectives

**Mathematical Perspective:**
If each query has recall R₁, R₂, ..., Rₙ, the combined recall approaches R₁ ∪ R₂ ∪ ... ∪ Rₙ, which is typically much larger than any individual Rᵢ.

🔍 Example from the arXiv dataset:
For the query “Photon production”, a single embedding query may miss the target paper
“Calculation of prompt diphoton production cross sections at Tevatron and LHC energies.”
However, a reformulated query like “diphoton production in hadron collisions” retrieves it successfully, demonstrating higher recall through query diversification.

✅ Conclusion:

This multi-query approach essentially casts a wider retrieval net, increasing the probability of capturing all relevant documents in the corpus.


## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [37]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = synthetic_arxiv_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [38]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [39]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [40]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [41]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [42]:
parent_document_retrieval_chain.invoke({"question" : "Which paper discusses diphoton production or cross sections in particle physics?"})["response"].content

'The paper that discusses diphoton production or cross sections in particle physics is titled "Calculation of prompt diphoton production cross sections at Tevatron and LHC energies" by C. Balázs, E. L. Berger, P. M. Nadolsky, and C.-P. Yuan.'

In [43]:
parent_document_retrieval_chain.invoke({"question" : "Find the paper that studies Stirling numbers or combinatorial determinants."})["response"].content

'Based on the provided information, there are two papers that study Stirling numbers or combinatorial determinants:\n\n1. **"A determinant of Stirling cycle numbers counts unlabeled acyclic single-source automata"** by David Callan.  \n   - Focuses on the determinant of Stirling cycle numbers and their combinatorial interpretation related to automata.\n\n2. **"Combinatorics and Boson normal ordering: A gentle introduction"** by P. Blasiak et al.  \n   - Discusses Stirling numbers in the context of operator ordering problems, relating combinatorial objects to these numbers.\n\nAdditionally, there is a paper titled **"Almost Product Evaluation of Hankel Determinants"** by Omer Egecioglu, Timothy Redmond, and Charles Ryavec, which studies Hankel determinants, some of which may involve combinatorial numbers like Stirling numbers, but the primary focus appears to be on the evaluation of Hankel determinants themselves.\n\n**Summary:**  \nThe most relevant paper studying Stirling numbers dire

In [44]:
parent_document_retrieval_chain.invoke({"question" : "Which paper investigates how the Moon’s orbit evolves over time?"})["response"].content

'The paper that investigates how the Moon’s orbit evolves over time is titled "The evolution of the Earth-Moon system based on the dark matter field fluid model" by Hongjun Pan.'

In [45]:
parent_document_retrieval_chain.invoke({"question" : "Which study connects harmonic analysis and Λα (Lambda-alpha) function spaces?"})["response"].content

'The study that connects harmonic analysis and Λα (Lambda-alpha) function spaces is titled "From dyadic Λα to Λα," authored by Wael Abu-Shammala and Alberto Torchinsky.'

In [46]:
parent_document_retrieval_chain.invoke({"question" : "What is the paper that introduces a sparsity-certifying algorithm for graph decomposition?"})["response"].content

'The paper that introduces a sparsity-certifying algorithm for graph decomposition is titled **"Sparsity-certifying Graph Decompositions"** by Ileana Streinu and Louis Theran.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [49]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [50]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [51]:
ensemble_retrieval_chain.invoke({"question" : "Which paper discusses diphoton production or cross sections in particle physics?"})["response"].content

'The paper that discusses diphoton production or cross sections in particle physics is titled:\n\n**"Calculation of prompt diphoton production cross sections at Tevatron and LHC energies"**\n\nIt presents a fully differential perturbative QCD calculation for the production of photon pairs at hadron colliders, including next-to-leading order contributions and resummation techniques. The abstract details their focus on diphoton distributions, agreement with Tevatron data, and predictions for LHC energies, explicitly addressing diphoton production cross sections.'

In [52]:
ensemble_retrieval_chain.invoke({"question" : "Find the paper that studies Stirling numbers or combinatorial determinants."})["response"].content

'Based on the provided documents, here are some papers that study Stirling numbers or combinatorial determinants:\n\n1. **"A determinant of Stirling cycle numbers counts unlabeled acyclic single-source automata" by David Callan**  \n   - Abstract: The paper shows that a determinant of Stirling cycle numbers counts unlabeled acyclic single-source automata, involving bijections to marked lattice paths and sign-reversing involutions.  \n   - Categories: math.CO\n\n2. **"Almost Product Evaluation of Hankel Determinants" by Omer Egecioglu, Timothy Redmond, Charles Ryavec**  \n   - Abstract: The authors evaluate Hankel determinants related to binomial entries, providing product and almost product formulas.  \n   - Categories: math.CO\n\n3. **"Determinant Formulas Relating to Tableaux of Bounded Height" by Guoce Xin**  \n   - Abstract: The paper develops determinant formulas for generating functions connected to oscillating tableaux and permutations with bounded increasing subsequences, which

In [53]:
ensemble_retrieval_chain.invoke({"question" : "Which paper investigates how the Moon’s orbit evolves over time?"})["response"].content

'The paper that investigates how the Moon’s orbit evolves over time is titled **"The evolution of the Earth-Moon system based on the dark matter field fluid model"** by Hongjun Pan. It discusses the orbital evolution of the Earth-Moon system, including the Moon\'s historical distance from Earth and how models explain its long-term behavior.'

In [54]:
ensemble_retrieval_chain.invoke({"question" : "Which study connects harmonic analysis and Λα (Lambda-alpha) function spaces?"})["response"].content

'The study that connects harmonic analysis and Lambda-alpha (Λα) function spaces is titled "From dyadic Λα to Λα" by Wael Abu-Shammala and Alberto Torchinsky. It discusses how to compute the Λα norm using the dyadic grid, which relates to harmonic analysis, and describes the Λα spaces in terms of dyadic and special atoms, thereby linking harmonic analysis to the study of these function spaces.'

In [55]:
ensemble_retrieval_chain.invoke({"question" : "What is the paper that introduces a sparsity-certifying algorithm for graph decomposition?"})["response"].content

'The paper that introduces a sparsity-certifying algorithm for graph decomposition is titled **"Sparsity-certifying Graph Decompositions"** by Ileana Streinu and Louis Theran.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [56]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [57]:
semantic_documents = semantic_chunker.split_documents(synthetic_arxiv_data[:20])

Let's create a new vector store.

In [58]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Arxiv_data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [59]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [60]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [61]:
semantic_retrieval_chain.invoke({"question" : "Which paper discusses diphoton production or cross sections in particle physics?"})["response"].content

'The paper that discusses diphoton production and cross sections in particle physics is titled "Calculation of prompt diphoton production cross sections at Tevatron and LHC energies" by C. Balázs, E. L. Berger, P. M. Nadolsky, and C.-P. Yuan.'

In [62]:
semantic_retrieval_chain.invoke({"question" : "Find the paper that studies Stirling numbers or combinatorial determinants."})["response"].content

'The paper that studies Stirling numbers or combinatorial determinants is titled "A determinant of Stirling cycle numbers counts unlabeled acyclic single-source automata" by David Callan. The abstract indicates that it involves determinants related to Stirling cycle numbers and their combinatorial counting interpretations.'

In [63]:
semantic_retrieval_chain.invoke({"question" : "Which paper investigates how the Moon’s orbit evolves over time?"})["response"].content

'The paper that investigates how the Moon’s orbit evolves over time is titled "The evolution of the Earth-Moon system based on the dark matter field fluid model" by Hongjun Pan.'

In [64]:
semantic_retrieval_chain.invoke({"question" : "Which study connects harmonic analysis and Λα (Lambda-alpha) function spaces?"})["response"].content

'The study that connects harmonic analysis and Λα (Lambda-alpha) function spaces is titled "From dyadic Λα to Λα" by Wael Abu-Shammala and Alberto Torchinsky.'

In [65]:
semantic_retrieval_chain.invoke({"question" : "What is the paper that introduces a sparsity-certifying algorithm for graph decomposition?"})["response"].content

'The paper that introduces a sparsity-certifying algorithm for graph decomposition is titled **"Sparsity-certifying Graph Decompositions"**, authored by Ileana Streinu and Louis Theran.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer

**How Semantic Chunking Would Behave with Short, Repetitive Sentences:**

**1. Problematic Behaviors:**
- **Over-fragmentation**: Short sentences would likely result in very small chunks, potentially creating one chunk per sentence or question-answer pair
- **High similarity scores**: Repetitive language patterns (e.g., "How do I...", "What is...", "Can I...") would create artificially high semantic similarity between unrelated topics
- **Poor breakpoint detection**: The algorithm might fail to identify meaningful topic boundaries because structural similarities overshadow content differences
- **Inconsistent chunking**: Similar FAQ formats might be grouped together regardless of their actual topic relevance

**2. Specific Issues:**
- **Template language interference**: Common FAQ phrases like "Please contact support" or "For more information" would create false semantic connections
- **Question format bias**: Questions with similar grammatical structures but different topics might be incorrectly grouped
- **Insufficient context**: Short sentences provide limited semantic information for meaningful similarity calculations

**Algorithm Adjustments:**

**1. Preprocessing Modifications:**
- **Content extraction**: Remove common FAQ template language and focus on core content words
- **Question-answer pairing**: Treat Q&A pairs as single semantic units rather than separate sentences
- **Keyword emphasis**: Weight domain-specific terms more heavily than structural/template words

**2. Threshold Adjustments:**
- **Lower similarity thresholds**: Use more stringent breakpoint thresholds to prevent over-grouping of structurally similar but topically different content
- **Adaptive thresholding**: Implement dynamic thresholds based on content length and repetitiveness metrics

**3. Enhanced Similarity Metrics:**
- **Content-focused embeddings**: Use embeddings that emphasize semantic content over syntactic structure
- **Topic modeling integration**: Incorporate topic modeling (e.g., LDA) to identify thematic boundaries beyond pure semantic similarity
- **Named entity recognition**: Weight entities and domain-specific terms more heavily in similarity calculations

**4. Structural Awareness:**
- **FAQ-specific chunking**: Implement FAQ-aware chunking that respects question-answer boundaries
- **Topic header detection**: Use FAQ section headers or categories as hard boundaries
- **Minimum chunk size**: Enforce minimum chunk sizes to prevent over-fragmentation

**5. Hybrid Approaches:**
- **Rule-based boundaries**: Combine semantic chunking with rule-based topic detection
- **Metadata utilization**: Leverage FAQ categories or tags if available
- **Manual seed boundaries**: Use predefined topic boundaries as starting points for semantic refinement

**Example Implementation Strategy:**
```
1. Preprocess: Remove template language, pair Q&As
2. Apply topic modeling to identify major themes
3. Use semantic chunking within topic boundaries
4. Enforce minimum chunk sizes (e.g., 3-5 Q&A pairs)
5. Post-process to merge overly small chunks
```

**Summary**:

Semantic chunking on repetitive data can mistake structure for meaning.
By preprocessing, adjusting thresholds, and combining semantic with rule-based cues, you can form larger, topic-coherent chunks and greatly improve retrieval quality.

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [ ]:
### YOUR CODE HERE